# Chapter 6 — CNN Architectures: VGG, ResNet, EfficientNet

## Learning Objectives
- Understand the design evolution: AlexNet → VGG → ResNet → EfficientNet
- Implement a ResNet residual block from scratch
- Understand skip connections and why they enable deeper networks
- Compare architectures on EuroSAT: parameters, FLOPS, accuracy
- Use torchvision.models with custom classification heads

## Estimated Duration: Theory 3h | Practical 2h | Total 5h
## Difficulty: Intermediate

## Architecture Timeline
  2012 AlexNet:      8 layers, 60M params, first GPU CNN, ImageNet winner
  2014 VGGNet:      19 layers, 138M params, uniform 3×3 convs
  2015 ResNet:      152 layers, 60M params, skip connections, ILSVRC winner
  2019 EfficientNet: compound scaling, 5-66M params, best accuracy/params

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q torch torchvision torchgeo matplotlib

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import models
from torchgeo.datasets import EuroSAT
import numpy as np
import matplotlib.pyplot as plt
import time
from pathlib import Path

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
FAST_MODE = DEVICE == 'cpu'
DATA_ROOT = Path('./data') if not IN_COLAB else Path('/content/data')
print(f'Device: {DEVICE} | Fast mode: {FAST_MODE}')

## 6.1 — The Degradation Problem and Skip Connections

The key problem with deep networks (before ResNet):
  Adding MORE layers made training WORSE — not due to overfitting,
  but due to optimisation difficulty (vanishing gradients).

He et al. (2015) insight: instead of learning H(x) directly,
learn the residual F(x) = H(x) - x.

Residual block:
  y = F(x) + x   where F(x) = Conv → BN → ReLU → Conv → BN

Why does this work?
  1. Gradient highway: gradients flow directly through skip connections
  2. Identity initialisation: if F(x)=0 initially, y=x (identity mapping)
  3. Easier to learn small residuals than full transformations
  4. Effectively creates an ensemble of networks of different depths

In [ ]:
# Implement a ResNet BasicBlock from scratch
class BasicBlock(nn.Module):
    """
    ResNet BasicBlock (used in ResNet-18 and ResNet-34).
    
    Architecture:
      x → Conv3×3 → BN → ReLU → Conv3×3 → BN → (+x) → ReLU → y
      
    If in_channels != out_channels (downsampling), a 1×1 conv
    on the skip path aligns the channel dimensions.
    """
    expansion = 1  # No bottleneck
    
    def __init__(self, in_channels: int, out_channels: int, stride: int = 1):
        super().__init__()
        
        # Main path: two 3×3 convolutions
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_channels)
        self.relu  = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_channels)
        
        # Skip path: 1×1 conv to match dimensions (if needed)
        self.downsample = None
        if stride != 1 or in_channels != out_channels:
            self.downsample = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )
    
    def forward(self, x):
        identity = x  # Save input for skip connection
        
        # Main path
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        
        # Skip connection (project identity if shape changed)
        if self.downsample is not None:
            identity = self.downsample(x)
        
        # Add residual: THIS IS THE KEY INNOVATION
        out = out + identity
        out = self.relu(out)
        return out


# Test the block
block = BasicBlock(64, 64)
x = torch.randn(4, 64, 32, 32)
y = block(x)
print(f'BasicBlock(64→64): input={x.shape} → output={y.shape}')

# Downsampling block
block_down = BasicBlock(64, 128, stride=2)
y_down = block_down(x)
print(f'BasicBlock(64→128, stride=2): input={x.shape} → output={y_down.shape}')
print(f'  Main path params: {sum(p.numel() for p in block_down.parameters()):,}')

In [ ]:
# Visualise gradient flow WITH vs WITHOUT skip connections
import torch.nn.functional as F

class PlainBlock(nn.Module):
    """Same as BasicBlock but without the skip connection."""
    def __init__(self, channels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels), nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
        )
    def forward(self, x): return F.relu(self.net(x))

class ResBlock(nn.Module):
    """BasicBlock WITH skip connection."""
    def __init__(self, channels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels), nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
        )
    def forward(self, x): return F.relu(self.net(x) + x)  # skip!

# Compare gradient norms at different depths
def measure_gradient_norms(block_class, n_blocks=20, channels=64):
    blocks = nn.Sequential(*[block_class(channels) for _ in range(n_blocks)])
    x = torch.randn(2, channels, 16, 16, requires_grad=False)
    
    # Forward pass with gradient hooks
    grad_norms = []
    hooks = []
    for block in blocks:
        def hook(module, grad_input, grad_output, _block=block):
            if grad_output[0] is not None:
                grad_norms.append(grad_output[0].norm().item())
        hooks.append(block.register_backward_hook(hook))
    
    out = blocks(x)
    loss = out.sum()
    loss.backward()
    
    for h in hooks:
        h.remove()
    
    return grad_norms[::-1]  # reverse to get layer order (input → output)

plain_grads = measure_gradient_norms(PlainBlock, n_blocks=20)
res_grads = measure_gradient_norms(ResBlock, n_blocks=20)

fig, ax = plt.subplots(figsize=(12, 5))
x_plot = range(1, min(len(plain_grads), len(res_grads)) + 1)
n = min(len(plain_grads), len(res_grads))
ax.semilogy(range(1, n+1), plain_grads[:n], label='Plain (no skip)', color='#F44336', linewidth=2)
ax.semilogy(range(1, n+1), res_grads[:n], label='Residual (with skip)', color='#2196F3', linewidth=2)
ax.set_xlabel('Layer (from input)')
ax.set_ylabel('Gradient Norm (log scale)')
ax.set_title('Gradient Flow: Plain Network vs ResNet\n'
             'Without skip connections, gradients vanish in early layers', fontsize=11)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Plain: grad norm at layer 1 = {plain_grads[0]:.2e}')
print(f'ResNet: grad norm at layer 1 = {res_grads[0]:.2e}')

In [ ]:
# Architecture comparison: parameters, speed, accuracy benchmark
from torchvision.models import (
    resnet18, ResNet18_Weights,
    resnet50, ResNet50_Weights,
    efficientnet_b0, EfficientNet_B0_Weights,
    efficientnet_b4, EfficientNet_B4_Weights,
    mobilenet_v3_small, MobileNet_V3_Small_Weights,
)

NUM_CLASSES = 10

def build_model(arch: str) -> nn.Module:
    if arch == 'resnet18':
        m = resnet18(weights=ResNet18_Weights.DEFAULT)
        m.fc = nn.Linear(512, NUM_CLASSES)
    elif arch == 'resnet50':
        m = resnet50(weights=ResNet50_Weights.DEFAULT)
        m.fc = nn.Linear(2048, NUM_CLASSES)
    elif arch == 'efficientnet_b0':
        m = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
        m.classifier[-1] = nn.Linear(1280, NUM_CLASSES)
    elif arch == 'efficientnet_b4':
        m = efficientnet_b4(weights=EfficientNet_B4_Weights.DEFAULT)
        m.classifier[-1] = nn.Linear(1792, NUM_CLASSES)
    elif arch == 'mobilenet_v3_small':
        m = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.DEFAULT)
        m.classifier[-1] = nn.Linear(1024, NUM_CLASSES)
    return m

architectures = ['resnet18', 'resnet50', 'efficientnet_b0', 'efficientnet_b4', 'mobilenet_v3_small']

print(f'{"Architecture":<22} {"Params":>10} {"Forward (ms)":>14} {"Memory (MB)":>12}')
print('-' * 62)

dummy_input = torch.randn(4, 3, 64, 64)  # EuroSAT batch

for arch in architectures:
    model = build_model(arch)
    params = sum(p.numel() for p in model.parameters())
    
    # Measure forward pass speed
    model.eval()
    with torch.no_grad():
        # Warm up
        _ = model(dummy_input)
        t0 = time.time()
        for _ in range(20):
            _ = model(dummy_input)
        ms_per_forward = (time.time() - t0) / 20 * 1000
    
    # Estimate memory (rough: params × 4 bytes for float32)
    mem_mb = params * 4 / 1e6
    
    print(f'{arch:<22} {params:>10,} {ms_per_forward:>12.1f}ms {mem_mb:>10.0f}MB')

In [ ]:
# Quick training comparison: ResNet18 vs EfficientNet-B0 on EuroSAT
EUROSAT_ROOT = DATA_ROOT / 'eurosat'

def collate_fn(batch):
    images = torch.stack([b['image'][:3].float() / 10000.0 for b in batch])
    labels = torch.tensor([b['label'] for b in batch])
    return images, labels

train_ds = EuroSAT(root=EUROSAT_ROOT, split='train', download=True)
val_ds   = EuroSAT(root=EUROSAT_ROOT, split='val',   download=True)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, collate_fn=collate_fn, num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False, collate_fn=collate_fn, num_workers=0)

N_EPOCHS_COMPARE = 3 if FAST_MODE else 10
criterion = nn.CrossEntropyLoss()

architectures_to_compare = ['resnet18'] if FAST_MODE else ['resnet18', 'efficientnet_b0']
compare_histories = {}

for arch in architectures_to_compare:
    print(f'\nTraining: {arch}')
    model = build_model(arch).to(DEVICE)
    opt = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    
    hist = {'train_acc': [], 'val_acc': []}
    for epoch in range(1, N_EPOCHS_COMPARE + 1):
        # Train
        model.train()
        correct = total = 0
        for imgs, lbls in train_loader:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            logits = model(imgs)
            loss = criterion(logits, lbls)
            loss.backward()
            opt.step()
            correct += (logits.argmax(1) == lbls).sum().item()
            total += len(imgs)
        train_acc = correct / total
        
        # Validate
        model.eval()
        correct = total = 0
        with torch.no_grad():
            for imgs, lbls in val_loader:
                imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
                logits = model(imgs)
                correct += (logits.argmax(1) == lbls).sum().item()
                total += len(imgs)
        val_acc = correct / total
        hist['train_acc'].append(train_acc)
        hist['val_acc'].append(val_acc)
        print(f'  Epoch {epoch}: train={train_acc:.3f}, val={val_acc:.3f}')
    
    compare_histories[arch] = hist

# Plot comparison
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#2196F3', '#F44336', '#4CAF50', '#FF9800']
for (arch, hist), color in zip(compare_histories.items(), colors):
    ep = range(1, len(hist['val_acc']) + 1)
    ax.plot(ep, [a*100 for a in hist['val_acc']], label=f'{arch} (val)', 
            color=color, linewidth=2)
    ax.plot(ep, [a*100 for a in hist['train_acc']], linestyle='--', 
            color=color, alpha=0.5, label=f'{arch} (train)')
ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy (%)'); ax.legend(fontsize=9)
ax.set_title('Architecture Comparison on EuroSAT (same lr, same data)', fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Practical Exercises

### Exercise 6.1 — ResNet Bottleneck Block
Implement the ResNet Bottleneck block used in ResNet-50/101/152:
  1×1 (reduce) → 3×3 (spatial) → 1×1 (expand)
expansion = 4: in_channels → in_channels//4 → in_channels//4 → in_channels
Verify that 2 BasicBlocks and 1 Bottleneck have similar parameter counts.

### Exercise 6.2 — Depth vs Width Ablation
Modify EuroSATCNN to:
  - Double the number of channels (wide)
  - Add 2 more conv blocks (deep)
  - Add skip connections to the deep version
Compare all three: do skip connections help with 5-block depth?

### Exercise 6.3 — Feature Visualisation
After training ResNet18, extract features from different layers:
  hooks on layer1, layer2, layer3, layer4
Run t-SNE on the extracted features and visualise class separation.
At which layer are the 10 classes best separated?

In [ ]:
print('Chapter 6 Summary:')
print('  ResNet: skip connections solve vanishing gradients for deep networks')
print('  Key formula: y = F(x) + x   (learn the residual, not the full mapping)')
print('  EfficientNet: compound scaling (depth × width × resolution simultaneously)')
print('  For EuroSAT: ResNet50 pretrained typically reaches >95% with fine-tuning')
print()
print('In Chapter 7: we explore WHY pretrained models work so well in EO')
print('and implement progressive fine-tuning strategies.')